# Claude Opus 4.6 Fast Cookbook

**Model:** anthropic/claude-opus-4.6-fast  
**Provider:** OpenRouter  
**Released:** April 7, 2026

Claude Opus 4.6 Fast is the fast-mode variant of Opus 4.6 with the same core capabilities and higher output speed at higher cost.

**Key specs from OpenRouter model page:**
- Context window: 1,000,000 tokens
- Max output: 128,000 tokens
- Starting input price: $30 per 1M tokens
- Starting output price: $150 per 1M tokens
- Fast mode note: higher throughput, premium pricing

**Table of Contents**
1. Setup and Installation
2. Basic Usage with OpenRouter
3. Framework Integration
4. Building Agents
5. Advanced Applications
6. Fast-Mode Operations and Benchmarking

## 1. Setup and Installation

In [ ]:
import subprocess
import sys

install_cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--disable-pip-version-check",
    "openai",
    "requests",
]

result = subprocess.run(install_cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("Required packages installed")
else:
    print("Package installation failed")
    if result.stderr:
        print(result.stderr.strip())

In [ ]:
import os
from typing import Dict, List, Any, Optional
from openai import OpenAI

MODEL = "anthropic/claude-opus-4.6-fast"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "your-openrouter-api-key")

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://github.com/your-org/your-app",
        "X-Title": "claude-opus-4.6-fast-cookbook",
    },
)

if OPENROUTER_API_KEY == "your-openrouter-api-key":
    print("OPENROUTER_API_KEY is not set. Update env var before running live calls.")
else:
    print("Client configured for model:", MODEL)

## 2. Basic Usage with OpenRouter

Start with a minimal chat completion call using the OpenAI-compatible API.

In [ ]:
def chat_once(user_message: str, temperature: float = 0.2, max_tokens: int = 1024) -> Dict[str, Any]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": user_message}],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return {
        "text": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else {},
        "id": response.id,
    }

print("chat_once helper ready")

In [ ]:
# Example live call (uncomment to run)
# result = chat_once("Write a concise migration plan for a legacy Python service to FastAPI.")
# print(result["text"])
# print(result["usage"])

print("Example cell ready")

In [ ]:
def stream_chat(user_message: str, temperature: float = 0.2):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": user_message}],
        temperature=temperature,
        stream=True,
    )

    chunks = []
    for chunk in stream:
        delta = chunk.choices[0].delta.content if chunk.choices else None
        if delta:
            print(delta, end="")
            chunks.append(delta)

    print()
    return "".join(chunks)

# Example:
# full_text = stream_chat("Summarize trade-offs between monolith and microservices.")

## 3. Framework Integration

Use OpenRouter with common orchestration frameworks when you need retrieval, routing, or multi-step chains.

In [ ]:
# LangChain example (optional dependency)
try:
    from langchain_openai import ChatOpenAI

    lc_model = ChatOpenAI(
        model=MODEL,
        base_url=OPENROUTER_BASE_URL,
        api_key=OPENROUTER_API_KEY,
        default_headers={
            "HTTP-Referer": "https://github.com/your-org/your-app",
            "X-Title": "claude-opus-4.6-fast-cookbook",
        },
    )

    response = lc_model.invoke("Generate 3 production hardening checks for a new API endpoint.")
    print(response.content)
except ImportError:
    print("Install langchain-openai to run this section: pip install langchain-openai")

## 4. Building Agents

This section shows a simple tool-calling loop for an agent workflow.

In [ ]:
import json

def get_weather(city: str) -> str:
    mock = {
        "san francisco": "16C, foggy",
        "new york": "22C, clear",
        "london": "14C, light rain",
    }
    return mock.get(city.lower(), "weather data unavailable")

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Return a mock weather string for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }
]

def run_tool_agent(user_prompt: str) -> str:
    messages: List[Dict[str, Any]] = [{"role": "user", "content": user_prompt}]

    first = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    assistant_message = first.choices[0].message
    messages.append(assistant_message.model_dump())

    if not assistant_message.tool_calls:
        return assistant_message.content or "No response"

    for tool_call in assistant_message.tool_calls:
        if tool_call.function.name == "get_weather":
            args = json.loads(tool_call.function.arguments)
            tool_result = get_weather(args["city"])
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result,
                }
            )

    second = client.chat.completions.create(
        model=MODEL,
        messages=messages,
    )
    return second.choices[0].message.content or "No final answer"

# Example:
# print(run_tool_agent("What is the weather in London and how should I dress?"))

## 5. Advanced Applications

For long contexts, chunk documents and summarize progressively before final synthesis.

In [ ]:
def split_text(text: str, chunk_size: int = 6000) -> List[str]:
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

def map_reduce_summary(document: str) -> str:
    chunks = split_text(document, chunk_size=6000)
    partials: List[str] = []

    for idx, chunk in enumerate(chunks, start=1):
        prompt = f"Summarize chunk {idx}/{len(chunks)} in 5 bullets:\n\n{chunk}"
        out = chat_once(prompt, temperature=0.1, max_tokens=900)["text"]
        partials.append(out)

    final_prompt = "Combine these chunk summaries into one executive summary with risks and actions:\n\n" + "\n\n".join(partials)
    return chat_once(final_prompt, temperature=0.1, max_tokens=1200)["text"]

print("map_reduce_summary ready")

## 6. Fast-Mode Operations and Benchmarking

Fast mode is useful for latency-sensitive workflows where you need Opus-level quality with faster output. Measure response time in your own workload before rollout.

In [ ]:
import time

def benchmark_latency(prompt: str, runs: int = 3) -> Dict[str, Any]:
    samples: List[float] = []

    for _ in range(runs):
        start = time.perf_counter()
        _ = chat_once(prompt, temperature=0.0, max_tokens=700)
        elapsed = time.perf_counter() - start
        samples.append(elapsed)

    return {
        "runs": runs,
        "min_seconds": min(samples),
        "max_seconds": max(samples),
        "avg_seconds": sum(samples) / len(samples),
        "samples": samples,
    }

# Example:
# metrics = benchmark_latency("Generate a checklist to triage API 500 errors.", runs=5)
# print(metrics)

## References
- OpenRouter model page: https://openrouter.ai/anthropic/claude-opus-4.6-fast
- OpenRouter docs: https://openrouter.ai/docs
- Anthropic fast mode docs: https://platform.claude.com/docs/en/build-with-claude/fast-mode

This notebook is designed for practical adaptation. Start with Section 2 and Section 6 to baseline quality, cost, and latency on your real prompts.